# Day 25: 实现分页分配、回收与 OOM

预先为请求预留容量，分配不足要原子失败。释放后的句柄失效，不能读到复用内存。教学实现显式 gather 页面，再做普通 Attention，不是 paged GPU kernel。

参考源码：`learning/mini_vllm/paged_kv_cache.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY22_30.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 25完整讲义](../docs/DAY25_OWNERSHIP_OOM.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

把free list看成可借抽屉账本：

```text
free=[3]，新请求要2块
    ↓ 先检查
拒绝：free仍是[3]，不登记半个请求，不动已有A/C的数据
```

请求B完成后，把它的块号放回free，旧B借阅证active=False。
C可以重用这些物理块，但C有效长度从0开始，
不能因为数组还残留旧数值就把它当成自己的历史。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(25, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

释放后旧引用还能读取吗？OOM 前后 free list 是否一致？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：请求结束显存没降一定泄漏吗？**
答：不一定，pool仍存在；先看页是否归还、live请求是否清掉。

**问：OOM前后最少比较什么？**
答：free、request表、已有状态及必要的实际storage，不只异常类型。

**问：有finally就等于生产取消完整了吗？**
答：不是，还需取消协议、pending工作、异步stream/引用生命周期与跨进程故障处理。

**问：旧值没清零可直接交给新请求吗？**
答：只有有效长度/写后读/所有权都正确才不会逻辑读旧值；这不保证安全擦除与多租户隔离。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY25_OWNERSHIP_OOM.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。